In [14]:
import pyspark
import sparkmonitor
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [16]:
listener_jar = Path(sparkmonitor.__path__[0]) / "listener_spark4_2.13.jar"
spark = (SparkSession.builder
    .config(
        "spark.extraListeners",
        "sparkmonitor.listener.JupyterSparkMonitorListener",
    )
    .config("spark.driver.extraClassPath", str(listener_jar))
    .appName("PySpark-Get-Started")
    .getOrCreate()
)


In [39]:
spark

In [73]:
df_csv = (
            spark.read.csv("data/orders.csv",
                           header=True,
                          # inferSchema=True
                          )
            # .repartition(2)
         )

# Example 1 without repartition

In [74]:
def sum_amount(col_name:str):
    return round(sum(col_name),2).alias(f"sum_{col_name}")

In [75]:
df_byCountry_amounts = (
    df_csv
    .filter(col("payment_method") == "Cash")
        .groupBy("country")
        .agg(sum_amount("price")
             ,sum_amount("discount")
            )
)

In [76]:
df_byCountry_amounts.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[country#1220], functions=[sum(cast(price#1216 as double)), sum(cast(discount#1222 as double))])
   +- Exchange hashpartitioning(country#1220, 200), ENSURE_REQUIREMENTS, [plan_id=827]
      +- HashAggregate(keys=[country#1220], functions=[partial_sum(cast(price#1216 as double)), partial_sum(cast(discount#1222 as double))])
         +- Project [price#1216, country#1220, discount#1222]
            +- Filter (isnotnull(payment_method#1221) AND (payment_method#1221 = Cash))
               +- FileScan csv [price#1216,country#1220,payment_method#1221,discount#1222] Batched: false, DataFilters: [isnotnull(payment_method#1221), (payment_method#1221 = Cash)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/ggrft/jupyter-lab/apache-spark/udemy/data/orders.csv], PartitionFilters: [], PushedFilters: [IsNotNull(payment_method), EqualTo(payment_method,Cash)], ReadSchema: struct<price:string,country:string,

In [77]:
df_byCountry_amounts.show()

+---------+---------+------------+
|  country|sum_price|sum_discount|
+---------+---------+------------+
|  Germany|  1370.37|      287.31|
|   France|  1790.83|      347.12|
|    India|  3051.86|      511.13|
|      USA|   2455.4|      389.46|
|       UK|  1335.64|      267.74|
|   Canada|  1827.38|      398.08|
|Australia|  1652.77|       389.8|
+---------+---------+------------+



# Example 2 with repartition

In [78]:
#Stage 1
# Wide Transformation
df = df_csv.repartition(2) 

#Stage 2
# Narrow Transformation - 1
df = df.select("order_id","customer_id")
# Narrow Transformation - 2
df = df.filter(col("order_id")==1001)
# End Stage 2

# Stage 3
# Wide Transformation
df = df.groupBy("customer_id").count()


In [79]:
df.show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|       C164|    1|
+-----------+-----+

